# TensorFlow Recommender Demo

This notebook is a lightweight companion to the main recommender example. Its purpose is to show practical familiarity with TensorFlow / Keras on a simple MovieLens embedding model, not to build a production recommender.

## Setup

- Uses the same MovieLens 100k files as the main recommender notebook.
- Dependencies are managed through the repository `requirements.txt`.
- The model is a small matrix-factorization style network built with user and item embeddings.

## What This Demo Shows

- A compact TensorFlow / Keras workflow for a simple embedding-based recommender.
- Explicit device detection so the notebook uses GPU if available and CPU otherwise.
- A time-aware `train -> validation -> test` split based on each eligible user's previous and latest favorable interactions (`rating >= 4`).
- Lightweight ranking training with sampled negatives and `Recall@10` evaluation.


In [49]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)
pd.set_option("display.max_colwidth", 60)

available_gpus = tf.config.list_physical_devices("GPU")
device_name = "GPU" if available_gpus else "CPU"
print(f"Running on: {device_name}")
if available_gpus:
    print(available_gpus)

Running on: CPU


## Data Loading

The notebook reuses `../data/u.data` and `../data/u.item`. If the files are missing, it downloads the official MovieLens 100k archive.

In [50]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

required_files = ["u.data", "u.item"]
if not all((data_dir / file_name).exists() for file_name in required_files):
    zip_path = data_dir / "ml-100k.zip"
    extract_dir = data_dir / "ml-100k"
    urlretrieve(MOVIELENS_URL, zip_path)
    with ZipFile(zip_path, "r") as zip_file:
        zip_file.extractall(data_dir)
    for file_name in required_files:
        source = extract_dir / file_name
        destination = data_dir / file_name
        if source.exists() and not destination.exists():
            destination.write_bytes(source.read_bytes())

ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item_id", "title"],
)

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings = ratings.merge(movies, on="item_id", how="left")
ratings.head()

,user_id,item_id,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)


## Preprocessing And Split

For consistency with the main notebook, we use a time-aware `train -> validation -> test` split. A user contributes to validation and test only if they have at least 3 total interactions, at least 2 favorable ratings (`rating >= 4`), and at least 1 interaction before the validation event.

Main slices used in this demo:
- `train_ratings`: earlier user history before the validation event
- `validation_ratings`: the second-latest favorable item per eligible user
- `test_ratings`: the latest favorable item per eligible user
- `train_positive`: favorable interactions from `train_ratings`
- `validation_positive` / `test_positive`: held-out favorable targets for ranking evaluation

How training examples are built:
- each observed favorable user-item interaction becomes a positive training example
- for the same user, we sample a few unseen items as negative training examples
- the model then learns to score the observed favorable item above those unseen items

This is why the deep demos use contrastive-style training while the main `TruncatedSVD` notebook does not: the neural ranking models need explicit positive and negative training examples, while `TruncatedSVD` learns from the interaction matrix directly.

This notebook uses contrastive-style training in a practical recommender sense: the model sees positive and negative user-item examples and learns to score the positives higher. It is simpler than formal contrastive learning setups, which usually compare paired representations directly and use specialized contrastive losses.

A more canonical modern ranking setup often uses pairwise losses or sampled-softmax / InfoNCE-style objectives rather than plain binary labels on sampled pairs. Those approaches are often stronger in larger systems, but this lighter pointwise setup is still common and defensible when the goal is a readable demo with a ranking objective.

Deep models train over many epochs, so they need a validation slice to monitor generalization during training. We use validation loss for early stopping, while `Recall@10` on held-out positives remains the ranking metric we care about.


In [51]:
POSITIVE_RATING = 4
MIN_USER_INTERACTIONS = 3
MIN_POSITIVE_INTERACTIONS = 2
NEGATIVES_PER_POSITIVE = 4
TOP_K = 10

user_ids = np.sort(ratings["user_id"].unique())
item_ids = np.sort(ratings["item_id"].unique())
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}

ratings["user_idx"] = ratings["user_id"].map(user_to_index)
ratings["item_idx"] = ratings["item_id"].map(item_to_index)


def make_time_split(frame, positive_rating=4, min_user_interactions=3, min_positive_interactions=2):
    frame = frame.sort_values(["user_id", "timestamp", "item_id"]).copy()
    train_parts = []
    validation_rows = []
    test_rows = []

    for _, user_history in frame.groupby("user_id", sort=False):
        positive_history = user_history[user_history["rating"] >= positive_rating]
        enough_history = len(user_history) >= min_user_interactions
        enough_positive = len(positive_history) >= min_positive_interactions

        if not (enough_history and enough_positive):
            train_parts.append(user_history)
            continue

        validation_row = positive_history.iloc[-2]
        test_row = positive_history.iloc[-1]
        train_end_idx = user_history.index.get_loc(validation_row.name)
        has_prior_history = train_end_idx > 0

        if not has_prior_history:
            train_parts.append(user_history)
            continue

        train_history = user_history.iloc[:train_end_idx]
        train_parts.append(train_history)
        validation_rows.append(validation_row)
        test_rows.append(test_row)

    train = pd.concat(train_parts, ignore_index=True)
    validation = pd.DataFrame(validation_rows).reset_index(drop=True)
    test = pd.DataFrame(test_rows).reset_index(drop=True)
    return train, validation, test


train_ratings, validation_ratings, test_ratings = make_time_split(
    ratings,
    positive_rating=POSITIVE_RATING,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_positive_interactions=MIN_POSITIVE_INTERACTIONS,
)

train_positive = train_ratings.loc[train_ratings["rating"] >= POSITIVE_RATING].reset_index(drop=True)
validation_positive = validation_ratings.reset_index(drop=True)
test_positive = test_ratings.reset_index(drop=True)
user_seen_items = train_ratings.groupby("user_id")["item_id"].apply(set).to_dict()


def build_pointwise_pairs(positive_frame, seen_lookup, all_item_ids, negatives_per_positive=4, seed=42):
    rng = np.random.default_rng(seed)
    all_item_ids = np.asarray(all_item_ids)
    user_indices = []
    item_indices = []
    labels = []

    for row in positive_frame.itertuples(index=False):
        seen_items = seen_lookup.get(row.user_id, set())
        negative_pool = all_item_ids[~np.isin(all_item_ids, list(seen_items))]
        if len(negative_pool) == 0:
            continue

        user_indices.append(row.user_idx)
        item_indices.append(row.item_idx)
        labels.append(1.0)

        sampled_negatives = rng.choice(
            negative_pool,
            size=min(negatives_per_positive, len(negative_pool)),
            replace=False,
        )
        for negative_item_id in sampled_negatives:
            user_indices.append(row.user_idx)
            item_indices.append(item_to_index[int(negative_item_id)])
            labels.append(0.0)

    return (
        np.asarray(user_indices, dtype=np.int32),
        np.asarray(item_indices, dtype=np.int32),
        np.asarray(labels, dtype=np.float32),
    )


train_user_idx, train_item_idx, train_labels = build_pointwise_pairs(
    train_positive,
    user_seen_items,
    item_ids,
    negatives_per_positive=NEGATIVES_PER_POSITIVE,
    seed=42,
)
validation_user_idx, validation_item_idx, validation_labels = build_pointwise_pairs(
    validation_positive,
    user_seen_items,
    item_ids,
    negatives_per_positive=NEGATIVES_PER_POSITIVE,
    seed=43,
)

print(f"Positive rating threshold: {POSITIVE_RATING}+")
print(f"Minimum total interactions for validation/test: {MIN_USER_INTERACTIONS}")
print(f"Minimum favorable interactions for validation/test: {MIN_POSITIVE_INTERACTIONS}")
print(f"Training positives: {len(train_positive):,}")
print(f"Validation positives: {len(validation_positive):,}")
print(f"Test positives: {len(test_positive):,}")
print(f"Training pairs after negative sampling: {len(train_labels):,}")
print(f"Validation pairs after negative sampling: {len(validation_labels):,}")


Positive rating threshold: 4+
Minimum total interactions for validation/test: 3
Minimum favorable interactions for validation/test: 2
Training positives: 53,491
Validation positives: 942
Test positives: 942
Training pairs after negative sampling: 267,455
Validation pairs after negative sampling: 4,710


## Model

We use a simple dot-product recommender with user and item bias terms. The model outputs a ranking logit: higher scores mean the model believes the user is more likely to engage with the item. For light regularization, TensorFlow uses layer-level `L2` penalties on embeddings and bias terms, while the PyTorch demo uses optimizer-level `weight_decay`. The exact implementation differs by framework, but the regularization intent is the same.

Compact model view:
- ranking logit: `s(u, i) = p_u^T q_i + b_u + b_i`
- probability used by the binary loss: `sigma(s(u, i))`
- regularization idea: keep embedding and bias weights from growing too large

Legend:
- `u`: user
- `i`: item
- `p_u`, `q_i`: user and item embedding vectors
- `b_u`, `b_i`: user and item bias terms
- `s(u, i)`: ranking logit before converting it into a probability
- `sigma(.)`: sigmoid function
- `batch`: the number of examples processed together in one training step
- `d`: embedding dimension

Why `Flatten()` appears here: in Keras, an embedding lookup with input shape `(1,)` returns a tensor shaped like `(batch, 1, d)`, where `batch` is the number of examples processed together and `d` is the embedding size. We flatten it to `(batch, d)` so the dot product operates on user and item vectors directly.


Why this differs from the main notebook: the main flow uses `TruncatedSVD`, a classical matrix-factorization baseline that learns from the interaction matrix directly and does not need sampled positive/negative training. These deep-learning demos use sampled positive/negative pairs because that is a more natural way to train a lightweight neural ranking model for `Recall@10`.


In [52]:
n_users = len(user_ids)
n_items = len(item_ids)
embedding_dim = 32
l2_strength = 1e-5

user_input = tf.keras.layers.Input(shape=(1,), name="user_idx")
item_input = tf.keras.layers.Input(shape=(1,), name="item_idx")

user_embedding = tf.keras.layers.Embedding(
    n_users,
    embedding_dim,
    name="user_embedding",
    embeddings_regularizer=tf.keras.regularizers.l2(l2_strength),
)(user_input)
item_embedding = tf.keras.layers.Embedding(
    n_items,
    embedding_dim,
    name="item_embedding",
    embeddings_regularizer=tf.keras.regularizers.l2(l2_strength),
)(item_input)
user_bias = tf.keras.layers.Embedding(
    n_users,
    1,
    name="user_bias",
    embeddings_regularizer=tf.keras.regularizers.l2(l2_strength),
)(user_input)
item_bias = tf.keras.layers.Embedding(
    n_items,
    1,
    name="item_bias",
    embeddings_regularizer=tf.keras.regularizers.l2(l2_strength),
)(item_input)

user_vector = tf.keras.layers.Flatten()(user_embedding)
item_vector = tf.keras.layers.Flatten()(item_embedding)
user_bias_term = tf.keras.layers.Flatten()(user_bias)
item_bias_term = tf.keras.layers.Flatten()(item_bias)
dot_product = tf.keras.layers.Dot(axes=1)([user_vector, item_vector])
score = tf.keras.layers.Add()([dot_product, user_bias_term, item_bias_term])

model = tf.keras.Model(inputs=[user_input, item_input], outputs=score)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
)
model.summary()


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_idx            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_idx            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │     30,176 │ user_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 32)     │     53,824 │ item_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_24          │ (None, 32)        │          0 │ user_embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_25          │ (None, 32)        │          0 │ item_embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_bias           │ (None, 1, 1)      │        943 │ user_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_bias           │ (None, 1, 1)      │      1,682 │ item_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_7 (Dot)         │ (None, 1)         │          0 │ flatten_24[0][0], │
│                     │                   │            │ flatten_25[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_26          │ (None, 1)         │          0 │ user_bias[0][0]   │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_27          │ (None, 1)         │          0 │ item_bias[0][0]   │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 1)         │          0 │ dot_7[0][0],      │
│                     │                   │            │ flatten_26[0][0], │
│                     │                   │            │ flatten_27[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 86,625 (338.38 KB)

 Trainable params: 86,625 (338.38 KB)

 Non-trainable params: 0 (0.00 B)

## Training

We allow up to 100 epochs and stop early based on validation performance. Here, `loss` is training binary cross-entropy on sampled positive/negative pairs, and `val_loss` is the same metric on the validation slice.

Stopping rule: train for at most 100 epochs, monitor `val_loss`, keep the best model state seen so far, and stop early if `val_loss` fails to improve for 5 consecutive epochs. That 5-epoch span is the patience window. So the run ends either because validation stops improving within that window or because it reaches the maximum epoch limit.

We train in batches so the model can update from many user-item examples at once. Smaller batches are noisier and cheaper, which can sometimes help generalization, while larger batches are smoother and more efficient but use more memory.


TensorFlow-specific implementation notes:

- **Training loop:** `model.fit(...)` handles the forward pass, loss computation, gradient updates, batching, and validation calls internally.
- **Validation phase:** when `validation_data=...` is passed into `fit(...)`, Keras evaluates validation separately from the training batches and tracks `val_loss` automatically.
- **Tracking metrics:** `history.history["loss"]` and `history.history["val_loss"]` store the training history after `fit(...)` finishes.
- **Early stopping:** `tf.keras.callbacks.EarlyStopping(...)` automates the same monitoring logic that the PyTorch notebook implements manually.
- **Restoring the best weights:** with `restore_best_weights=True`, Keras automatically puts the model back at the best validation state before downstream evaluation and recommendation.


In [53]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    x=[train_user_idx, train_item_idx],
    y=train_labels,
    validation_data=([validation_user_idx, validation_item_idx], validation_labels),
    batch_size=2048,
    epochs=100,
    callbacks=[early_stopping],
    verbose=0,
)

pd.DataFrame(history.history)


,loss,val_loss
0,0.666205,0.645330
1,0.560669,0.527710
2,0.410219,0.467293
3,0.375854,0.457052
4,0.369787,0.453396
5,0.366995,0.451017
6,0.364495,0.448976
7,0.361799,0.447040
8,0.358896,0.445149
9,0.355890,0.443295


## Evaluation

We report `Recall@10` on held-out favorable items. With one held-out favorable item per user, `Recall@10` is simply the share of evaluated users whose held-out item appears in the top 10 recommendations.


Evaluation note: for each user, the notebook scores the full candidate item set in one vectorized pass, masks items already seen in history, and then checks whether the held-out favorable item appears in the top `K`.


Quick reading guide: if `loss` and `val_loss` stay high while `Recall@10` stays low, the model may be underfitting. If training `loss` keeps falling but `val_loss` stops improving and `Recall@10` stays weak, the model may be overfitting. The healthiest pattern is lower validation loss together with stronger `Recall@10`.


In [54]:
def score_all_items_for_user(user_id):
    user_idx = user_to_index[user_id]
    # `np.arange(...)` creates the full item index list, which means items = [0, 1, 2, ...].
    # `np.full(...)` repeats the same user id, which means user = [u, u, u, ...].
    # Together, the model scores (u, 0), (u, 1), (u, 2), ... in one pass.
    all_item_indices = np.arange(n_items, dtype=np.int32).reshape(-1, 1)
    user_array = np.full(shape=(n_items, 1), fill_value=user_idx, dtype=np.int32)
    scores = model.predict([user_array, all_item_indices], verbose=0).reshape(-1)
    return scores


def recall_at_k(eval_frame, seen_frame, k=10):
    seen_lookup = seen_frame.groupby("user_id")["item_idx"].apply(set).to_dict()
    hits = []

    for row in eval_frame.itertuples(index=False):
        scores = score_all_items_for_user(row.user_id)
        seen_indices = np.fromiter(seen_lookup.get(row.user_id, set()), dtype=np.int32)
        if len(seen_indices) > 0:
            scores[seen_indices] = -np.inf

        top_indices = np.argsort(scores)[::-1][:k]
        hits.append(int(row.item_idx in top_indices))

    return float(np.mean(hits))


validation_recall = recall_at_k(validation_positive, train_ratings, k=TOP_K)
test_history = pd.concat([train_ratings, validation_ratings], ignore_index=True)
test_recall = recall_at_k(test_positive, test_history, k=TOP_K)

print(f"Validation Recall@{TOP_K}: {validation_recall:.4f}")
print(f"Test Recall@{TOP_K}: {test_recall:.4f}")


Validation Recall@10: 0.1295
Test Recall@10: 0.0998


Interpretation: higher `Recall@10` means the held-out favorable item appears more often near the top of the recommendation list. This is a ranking metric, so score ordering matters more than score calibration.


## Example Recommendations

Below we score all unseen movies for one user and return the top recommendations.

In [55]:
movie_lookup = movies.set_index("item_id")

def recommend_for_user(user_id, seen_frame, k=10):
    scores = score_all_items_for_user(user_id)
    seen_indices = seen_frame.loc[seen_frame["user_id"] == user_id, "item_idx"].to_numpy()
    scores[seen_indices] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]
    top_item_ids = [item_ids[idx] for idx in top_indices]
    return pd.DataFrame(
        {
            "item_id": top_item_ids,
            "title": movie_lookup.loc[top_item_ids, "title"].values,
            "ranking_score": scores[top_indices],
        }
    )

example_user_id = int(test_positive.iloc[0]["user_id"])
example_seen = pd.concat([train_ratings, validation_ratings], ignore_index=True)
recommend_for_user(example_user_id, example_seen, k=10)


,item_id,title,ranking_score
0,475,Trainspotting (1996),2.701177
1,357,One Flew Over the Cuckoo's Nest (1975),2.353123
2,318,Schindler's List (1993),2.288236
3,474,Dr. Strangelove or: How I Learned to Stop Worrying and L...,2.124234
4,423,E.T. the Extra-Terrestrial (1982),2.025991
5,288,Scream (1996),1.915029
6,433,Heathers (1989),1.882257
7,276,Leaving Las Vegas (1995),1.834395
8,655,Stand by Me (1986),1.832745
9,483,Casablanca (1942),1.800198


## What This Demo Does Not Cover

- Production-scale candidate generation, ranking, and re-ranking.
- Rich side features such as genres, user metadata, or sequence context.
- ANN retrieval, online testing, or production monitoring.
- A deeper comparison against stronger recommender baselines.
